In [1]:

from pathlib import Path
import xml.etree.ElementTree as ET
import pandas as pd

# Define project data directory
DATA_DIR = Path(
    r"C:\Users\bucco\Desktop\Laboratory of Data Science"
    r"\project\code_project\data\Project_Data"
)

EVENTS_PATH = DATA_DIR / "subsample_events.xml"

if not EVENTS_PATH.is_file():
    raise FileNotFoundError(f"File not found: {EVENTS_PATH}")

# Parse XML records
def read_events_xml(path):
    records = []

    context = ET.iterparse(path, events=("start", "end"))
    _, root = next(context)

    for event, element in context:
        if event == "end" and element.tag == "row":
            records.append({
                child.tag: child.text
                for child in element
            })
            root.clear()

    return pd.DataFrame.from_records(records)


# Preserve the original dataset
events_raw = read_events_xml(EVENTS_PATH)

# Create a working copy for cleaning
events_clean = events_raw.copy()

print(f"Original records: {len(events_raw):,}")
print(f"Working records: {len(events_clean):,}")
print(f"Columns: {len(events_clean.columns)}")
print("Dataset successfully loaded.")


Original records: 716,643
Working records: 716,643
Columns: 6
Dataset successfully loaded.


In [2]:

# ============================================================
# STEP 1: STANDARDIZE DATA TYPES AND MISSING VALUES
# ============================================================

# Standardize text fields and missing value representations
for column in events_clean.columns:
    events_clean[column] = (
        events_clean[column]
        .astype("string")
        .str.strip()
        .replace("", pd.NA)
    )

# Convert identifier columns to nullable integers
id_columns = ["index", "event_id", "truth_id", "user_id"]

for column in id_columns:
    numeric_values = pd.to_numeric(
        events_clean[column],
        errors="coerce"
    )

    # Preserve only valid integer values
    valid_integers = (
        numeric_values.notna()
        & numeric_values.mod(1).eq(0)
    )

    events_clean[column] = (
        numeric_values
        .where(valid_integers)
        .astype("Int64")
    )

# Convert timestamps to datetime
events_clean["timestamp"] = pd.to_datetime(
    events_clean["timestamp"],
    errors="coerce",
    utc=True,
    format="mixed"
)

# Standardize event types
events_clean["event_type"] = (
    events_clean["event_type"]
    .str.upper()
)

# Verify cleaning transformations
print("Data types after standardization:")
print(events_clean.dtypes)

print("\nRecords preserved:", len(events_clean))
print("Columns preserved:", len(events_clean.columns))


Data types after standardization:
index                       Int64
event_id                    Int64
event_type                 string
truth_id                    Int64
user_id                     Int64
timestamp     datetime64[ns, UTC]
dtype: object

Records preserved: 716643
Columns preserved: 6


In [3]:

# ============================================================
# STEP 2: VALIDATE DATA TYPE CONVERSIONS
# ============================================================

conversion_issues = []

for column in events_clean.columns:

    # Identify non-empty values in the original dataset
    original_values = events_raw[column].astype("string").str.strip()

    original_valid = (
        original_values.notna()
        & original_values.ne("")
    )

    # Identify values lost during conversion
    lost_values = original_valid & events_clean[column].isna()

    conversion_issues.append({
        "column": column,
        "values_before_conversion": int(original_valid.sum()),
        "values_lost": int(lost_values.sum()),
        "loss_percentage": round(
            lost_values.sum() / len(events_raw) * 100, 4
        )
    })

conversion_report = pd.DataFrame(conversion_issues)

display(conversion_report)

print(
    "\nTotal values lost during conversion:",
    conversion_report["values_lost"].sum()
)


,column,values_before_conversion,values_lost,loss_percentage
0,index,716643,0,0.0
1,event_id,716643,0,0.0
2,event_type,716643,0,0.0
3,truth_id,716643,0,0.0
4,user_id,716643,0,0.0
5,timestamp,716643,0,0.0



Total values lost during conversion: 0


In [4]:

# ============================================================
# STEP 3: VALIDATE SPECIAL IDENTIFIER VALUES
# ============================================================

id_columns = ["index", "event_id", "truth_id", "user_id"]

# Identify invalid or special identifier values
identifier_report = []

for column in id_columns:

    identifier_report.append({
        "column": column,
        "missing_values": int(events_clean[column].isna().sum()),
        "negative_values": int((events_clean[column] < 0).sum()),
        "zero_values": int((events_clean[column] == 0).sum())
    })

identifier_report = pd.DataFrame(identifier_report)

display(identifier_report)

# Inspect event types associated with user_id = -1
unknown_user_events = (
    events_clean.loc[
        events_clean["user_id"] == -1,
        "event_type"
    ]
    .value_counts()
    .rename_axis("event_type")
    .reset_index(name="records")
)

print("\nEvent types associated with user_id = -1:")
display(unknown_user_events)


,column,missing_values,negative_values,zero_values
0,index,0,0,1
1,event_id,0,0,0
2,truth_id,0,0,0
3,user_id,0,106949,0



Event types associated with user_id = -1:


,event_type,records
0,LI,68553
1,RET,25828
2,REP,12568


In [5]:

# ============================================================
# STEP 4: HANDLE UNKNOWN USER IDENTIFIERS
# ============================================================

# Identify sentinel values for unknown users
unknown_user_mask = events_clean["user_id"] == -1

# Preserve information about unknown users
events_clean["user_id_unknown"] = unknown_user_mask.astype("boolean")

# Replace sentinel values with proper missing values
events_clean.loc[unknown_user_mask, "user_id"] = pd.NA

# Validate the transformation
print("Unknown user identifiers:", unknown_user_mask.sum())

print(
    "Missing user IDs after cleaning:",
    events_clean["user_id"].isna().sum()
)

print(
    "Unknown user flags:",
    events_clean["user_id_unknown"].sum()
)

print(
    "Records preserved:",
    len(events_clean)
)

assert events_clean["user_id"].isna().sum() == unknown_user_mask.sum()
assert len(events_clean) == len(events_raw)

print("\nUnknown user identifiers successfully standardized.")


Unknown user identifiers: 106949
Missing user IDs after cleaning: 106949
Unknown user flags: 106949
Records preserved: 716643

Unknown user identifiers successfully standardized.


In [6]:

# ============================================================
# STEP 5: INSPECT USERS XML RECORD STRUCTURE
# ============================================================

USERS_PATH = DATA_DIR / "users.xml"

if not USERS_PATH.is_file():
    raise FileNotFoundError(f"File not found: {USERS_PATH}")

# Inspect the first complete user record
context = ET.iterparse(USERS_PATH, events=("start", "end"))

_, root = next(context)

print("Root element:", root.tag)

for event, element in context:

    # Stop when the first direct child of the root is complete
    if event == "end" and element in root:

        print("\nUser record element:", element.tag)
        print("Attributes:", element.attrib)

        print("\nFields:")
        for child in element:
            print(f"  {child.tag}: {child.text}")

        break


Root element: data

User record element: row
Attributes: {}

Fields:
  id_user: 1238
  timestamp_user_creation: 2022-02-01 00:00:00
  username: jeffgolfs
  follower_count: 886
  following_count: 204
  profile_url: https://truthsocial.com/@jeffgolfs
  finished_follower_scrape: f
  finished_following_scrape: f
  finished_truth_scrape: f


In [7]:

# ============================================================
# STEP 6: VALIDATE USER IDS AGAINST USERS.XML
# ============================================================

# Extract user identifiers from users.xml
user_ids = []

context = ET.iterparse(USERS_PATH, events=("start", "end"))
_, root = next(context)

for event, element in context:
    if event == "end" and element.tag == "row":

        user_id = element.findtext("id_user")
        user_ids.append(user_id)

        root.clear()

# Convert reference identifiers to nullable integers
users_ids = pd.to_numeric(
    pd.Series(user_ids, dtype="string"),
    errors="coerce"
).astype("Int64")

# Create the reference set
valid_user_ids = set(users_ids.dropna())

# Check referential integrity
known_user_mask = events_clean["user_id"].notna()

matched_user_mask = (
    known_user_mask
    & events_clean["user_id"].isin(valid_user_ids)
)

unmatched_user_mask = (
    known_user_mask
    & ~events_clean["user_id"].isin(valid_user_ids)
)

# Generate validation report
user_integrity_report = pd.DataFrame({
    "status": [
        "Matched user IDs",
        "Unmatched user IDs",
        "Unknown user IDs"
    ],
    "records": [
        int(matched_user_mask.sum()),
        int(unmatched_user_mask.sum()),
        int(events_clean["user_id"].isna().sum())
    ]
})

display(user_integrity_report)

print("\nUsers in reference dataset:", len(valid_user_ids))
print("Duplicate user IDs in reference:", users_ids.dropna().duplicated().sum())
print("Invalid user IDs in reference:", users_ids.isna().sum())
print("Total events:", len(events_clean))

assert user_integrity_report["records"].sum() == len(events_clean)


,status,records
0,Matched user IDs,609694
1,Unmatched user IDs,0
2,Unknown user IDs,106949



Users in reference dataset: 487292
Duplicate user IDs in reference: 0
Invalid user IDs in reference: 0
Total events: 716643


In [8]:

# ============================================================
# STEP 7: VALIDATE TRUTH IDS AGAINST TRUTHS.CSV
# ============================================================

TRUTHS_PATH = DATA_DIR / "truths.csv"

if not TRUTHS_PATH.is_file():
    raise FileNotFoundError(f"File not found: {TRUTHS_PATH}")

# Load only the post identifiers
truths_ids = pd.read_csv(
    TRUTHS_PATH,
    usecols=["id"],
    dtype={"id": "string"}
)["id"]

# Convert reference identifiers
truths_ids = pd.to_numeric(
    truths_ids,
    errors="coerce"
).astype("Int64")

# Build reference set
valid_truth_ids = set(truths_ids.dropna())

# Check foreign key correspondence
matched_truth_mask = events_clean["truth_id"].isin(valid_truth_ids)

unmatched_truth_mask = (
    events_clean["truth_id"].notna()
    & ~matched_truth_mask
)

# Generate validation report
truth_integrity_report = pd.DataFrame({
    "status": [
        "Matched truth IDs",
        "Unmatched truth IDs",
        "Missing truth IDs"
    ],
    "records": [
        int(matched_truth_mask.sum()),
        int(unmatched_truth_mask.sum()),
        int(events_clean["truth_id"].isna().sum())
    ]
})

display(truth_integrity_report)

print("\nUnique truths in reference:", len(valid_truth_ids))
print("Duplicate truth IDs in reference:", truths_ids.dropna().duplicated().sum())
print("Invalid truth IDs in reference:", truths_ids.isna().sum())
print("Total events:", len(events_clean))

assert truth_integrity_report["records"].sum() == len(events_clean)


,status,records
0,Matched truth IDs,483785
1,Unmatched truth IDs,232858
2,Missing truth IDs,0



Unique truths in reference: 1038821
Duplicate truth IDs in reference: 12328
Invalid truth IDs in reference: 0
Total events: 716643


In [9]:

# ============================================================
# STEP 8: INSPECT ADDITIONAL TRUTHS SCHEMA
# ============================================================

ADDITIONAL_TRUTHS_PATH = (
    DATA_DIR.parent
    / "Additional_Data"
    / "additional_truths.arff"
)

if not ADDITIONAL_TRUTHS_PATH.is_file():
    raise FileNotFoundError(
        f"File not found: {ADDITIONAL_TRUTHS_PATH}"
    )

# Read only the ARFF header
attributes = []

with open(
    ADDITIONAL_TRUTHS_PATH,
    "r",
    encoding="utf-8"
) as file:

    for line in file:
        line = line.strip()

        if line.lower().startswith("@attribute"):
            attributes.append(line)

        elif line.lower() == "@data":
            break

print("Attributes in additional_truths.arff:\n")

for attribute in attributes:
    print(attribute)

print("\nTotal attributes:", len(attributes))


Attributes in additional_truths.arff:

@attribute id integer
@attribute timestamp string
@attribute text string
@attribute url string
@attribute reply_count real
@attribute retruth_count real
@attribute like_count real
@attribute external_id string
@attribute author real
@attribute time_scraped string

Total attributes: 10


In [10]:

import csv

# ============================================================
# STEP 9: RECOVER TRUTH REFERENCES FROM ADDITIONAL DATA
# ============================================================

additional_truth_ids = set()
invalid_additional_ids = 0
total_additional_records = 0
data_started = False

with open(ADDITIONAL_TRUTHS_PATH, "r", encoding="utf-8") as file:

    for line in file:
        line = line.strip()

        if not data_started:
            if line.lower() == "@data":
                data_started = True
            continue

        # Skip empty lines and ARFF comments
        if not line or line.startswith("%"):
            continue

        total_additional_records += 1

        # Read the first field, corresponding to 'id'
        fields = next(csv.reader([line]))

        try:
            additional_truth_ids.add(int(fields[0].strip()))
        except (ValueError, IndexError):
            invalid_additional_ids += 1

# Identify events recoverable through the additional dataset
recovered_truth_mask = (
    unmatched_truth_mask
    & events_clean["truth_id"].isin(additional_truth_ids)
)

still_unmatched_mask = (
    unmatched_truth_mask
    & ~events_clean["truth_id"].isin(additional_truth_ids)
)

# Generate recovery report
truth_recovery_report = pd.DataFrame({
    "status": [
        "Originally matched in truths.csv",
        "Recovered through additional_truths.arff",
        "Still unmatched"
    ],
    "records": [
        int(matched_truth_mask.sum()),
        int(recovered_truth_mask.sum()),
        int(still_unmatched_mask.sum())
    ]
})

display(truth_recovery_report)

print("\nAdditional truth records:", total_additional_records)
print("Unique additional truth IDs:", len(additional_truth_ids))
print("Invalid additional IDs:", invalid_additional_ids)

print(
    "\nTotal events:",
    len(events_clean)
)

assert truth_recovery_report["records"].sum() == len(events_clean)


,status,records
0,Originally matched in truths.csv,483785
1,Recovered through additional_truths.arff,232858
2,Still unmatched,0



Additional truth records: 35222
Unique additional truth IDs: 35222
Invalid additional IDs: 0

Total events: 716643


In [11]:

# ============================================================
# STEP 11: VALIDATE DUPLICATE EVENTS
# ============================================================

# Identify duplicated event identifiers
duplicate_id_mask = events_clean.duplicated(
    subset=["event_id"],
    keep=False
)

# Identify completely identical events
event_columns = [
    "event_id",
    "event_type",
    "truth_id",
    "user_id",
    "timestamp"
]

exact_duplicate_mask = events_clean.duplicated(
    subset=event_columns,
    keep=False
)

# Check whether the same event_id has conflicting information
conflicting_events = (
    events_clean.loc[duplicate_id_mask, event_columns]
    .groupby("event_id", dropna=False)
    .nunique(dropna=False)
)

conflicting_event_ids = conflicting_events.index[
    conflicting_events.gt(1).any(axis=1)
]

print("Records with duplicated event_id:", duplicate_id_mask.sum())

print("Records belonging to identical event groups:",
      exact_duplicate_mask.sum())

print("Event IDs with conflicting information:",
      len(conflicting_event_ids))

print("\nExamples of conflicting events:")

display(
    events_clean[
        events_clean["event_id"].isin(conflicting_event_ids)
    ][event_columns].head(10)
)


Records with duplicated event_id: 0
Records belonging to identical event groups: 0
Event IDs with conflicting information: 0

Examples of conflicting events:


,event_id,event_type,truth_id,user_id,timestamp


In [12]:

# ============================================================
# STEP 12: VALIDATE EVENT TYPE DOMAIN
# ============================================================

# Define the expected event types
valid_event_types = {"LI", "RET", "REP"}

# Identify invalid or missing event types
invalid_event_type_mask = (
    events_clean["event_type"].isna()
    | ~events_clean["event_type"].isin(valid_event_types)
)

invalid_event_count = int(invalid_event_type_mask.sum())

print("Invalid or missing event types:", invalid_event_count)

if invalid_event_count > 0:
    print("\nUnexpected event type values:")

    display(
        events_clean.loc[
            invalid_event_type_mask,
            "event_type"
        ].value_counts(dropna=False)
    )
else:
    print("All event types belong to the expected domain.")

# Validate record preservation
assert len(events_clean) == len(events_raw)

print("\nRecords preserved:", len(events_clean))


Invalid or missing event types: 2419

Unexpected event type values:


event_type
CR    2419
Name: count, dtype: Int64


Records preserved: 716643


In [13]:

# ============================================================
# STEP 13: VALIDATE COMPLETE EVENT TYPE DOMAIN
# ============================================================

# Include all observed event types
valid_event_types = {"LI", "RET", "REP", "CR"}

# Identify unexpected or missing values
invalid_event_type_mask = (
    events_clean["event_type"].isna()
    | ~events_clean["event_type"].isin(valid_event_types)
)

invalid_event_count = int(invalid_event_type_mask.sum())

print("Invalid or missing event types:", invalid_event_count)

if invalid_event_count > 0:
    display(
        events_clean.loc[
            invalid_event_type_mask,
            "event_type"
        ].value_counts(dropna=False)
    )
else:
    print("All event types belong to the observed domain.")

# Ensure no records have been removed
assert len(events_clean) == len(events_raw)

print("\nRecords preserved:", len(events_clean))


Invalid or missing event types: 0
All event types belong to the observed domain.

Records preserved: 716643


In [14]:

# ============================================================
# STEP 14: VALIDATE TEMPORAL CONSISTENCY
# ============================================================

# Truth Social launched in February 2022
platform_start = pd.Timestamp("2022-02-21", tz="UTC")

# Identify missing timestamps
missing_timestamp_mask = events_clean["timestamp"].isna()

# Identify events before the platform launch
before_platform_mask = (
    events_clean["timestamp"].notna()
    & (events_clean["timestamp"] < platform_start)
)

# Generate temporal validation report
temporal_report = pd.DataFrame({
    "issue": [
        "Missing timestamps",
        "Events before platform launch"
    ],
    "records": [
        int(missing_timestamp_mask.sum()),
        int(before_platform_mask.sum())
    ]
})

display(temporal_report)

# Inspect only potentially inconsistent records
if before_platform_mask.any():
    print("\nExamples of potentially inconsistent events:")

    display(
        events_clean.loc[
            before_platform_mask,
            [
                "event_id",
                "event_type",
                "truth_id",
                "timestamp"
            ]
        ].head(10)
    )

print("\nRecords preserved:", len(events_clean))


,issue,records
0,Missing timestamps,0
1,Events before platform launch,51



Examples of potentially inconsistent events:


,event_id,event_type,truth_id,timestamp
459384,30009021,LI,366452,2022-02-16 13:43:51.871948115+00:00
474706,101746,CR,719187,2022-02-14 00:00:00.000000000+00:00
481255,28725598,LI,325285,2022-02-16 17:10:51.707788815+00:00
482909,7949458,LI,896293,2022-02-16 00:27:12.593851010+00:00
486862,33795581,LI,604634,2022-02-17 10:37:00.491020770+00:00
488731,34871804,RET,758728,2022-02-16 14:51:04.087105883+00:00
497433,30772,RET,807654,2022-02-17 15:16:13.872124951+00:00
502076,34877790,REP,758728,2022-02-20 12:38:47.653886370+00:00
502202,29888680,LI,366261,2022-02-19 18:06:27.482201443+00:00
504166,1249888,LI,751049,2022-02-17 14:19:33.546119352+00:00



Records preserved: 716643


In [15]:

# ============================================================
# STEP 15: ASSESS DETERMINISTIC USER ID RECOVERY
# ============================================================

# Separate events with known and unknown user IDs
known_events = events_clean[
    events_clean["user_id"].notna()
].copy()

unknown_events = events_clean[
    events_clean["user_id"].isna()
].copy()

# Build an event_id -> user_id lookup
event_user_lookup = (
    known_events
    .set_index("event_id")["user_id"]
)

# Attempt deterministic recovery through event_id
recovered_users = unknown_events["event_id"].map(
    event_user_lookup
)

recoverable_count = int(recovered_users.notna().sum())

print("Unknown user IDs:", len(unknown_events))
print("Recoverable user IDs through event_id:", recoverable_count)
print(
    "Still unresolved:",
    len(unknown_events) - recoverable_count
)

# No modifications are applied at this stage
assert len(events_clean) == len(events_raw)


Unknown user IDs: 106949
Recoverable user IDs through event_id: 0
Still unresolved: 106949


In [16]:

# ============================================================
# STEP 16: INSPECT QUOTES DATA FOR USER ID RECOVERY
# ============================================================

import json

QUOTES_PATH = DATA_DIR / "quotes.json"

if not QUOTES_PATH.is_file():
    raise FileNotFoundError(f"File not found: {QUOTES_PATH}")

# Load quotes dataset
with open(QUOTES_PATH, "r", encoding="utf-8") as file:
    quotes_data = json.load(file)

# Inspect top-level structure
print("JSON structure:", type(quotes_data).__name__)

if isinstance(quotes_data, list):
    print("Total records:", len(quotes_data))

    if quotes_data:
        print("\nAvailable fields:")
        print(list(quotes_data[0].keys()))

        print("\nFirst record:")
        display(quotes_data[0])

elif isinstance(quotes_data, dict):
    print("\nTop-level keys:")
    print(list(quotes_data.keys()))

    print("\nFirst entries:")
    display(dict(list(quotes_data.items())[:3]))


JSON structure: dict

Top-level keys:
['timestamp', 'quoted_user', 'quoting_user', 'quoting_truth', 'quoted_truth_external_id']

First entries:


{'timestamp': {'0': '2022-07-22 09:43:00',
  '1': '2022-08-20 18:00:00',
  '2': '2022-07-14 21:13:00',
  '3': '2022-09-12 22:40:00',
  '4': '2022-11-11 00:00:00',
  '5': '2022-09-12 21:49:00',
  '6': '2022-09-12 21:44:00',
  '7': '2022-09-07 15:01:00',
  '8': '2022-09-07 15:00:00',
  '9': '2022-09-05 18:07:00',
  '10': '2022-09-05 16:44:00',
  '11': '2022-09-05 14:36:00',
  '12': '2022-09-05 14:34:00',
  '13': '2022-09-02 03:25:00',
  '14': '2022-09-02 03:23:00',
  '15': '2022-09-02 03:18:00',
  '16': '2022-09-02 02:48:00',
  '17': '2022-09-02 02:44:00',
  '18': '2022-08-30 08:54:00',
  '19': '2022-08-30 08:52:00',
  '20': '2022-08-30 08:49:00',
  '21': '2022-08-30 08:39:00',
  '22': '2022-08-30 08:29:00',
  '23': '2022-08-30 08:26:00',
  '24': '2022-08-30 08:23:00',
  '25': '2022-08-30 08:22:00',
  '26': '2022-08-30 08:18:00',
  '27': '2022-08-30 08:16:00',
  '28': '2022-08-30 08:05:00',
  '29': '2022-08-30 08:02:00',
  '30': '2022-07-01 16:34:00',
  '31': '2022-08-30 07:49:00',
  '32

In [17]:

# ============================================================
# STEP 17: ASSESS QUOTE-BASED USER RECOVERY
# ============================================================

# Convert column-oriented JSON into a DataFrame
quotes_df = pd.DataFrame(quotes_data)

# Standardize potential join keys
quotes_df["quoting_truth"] = pd.to_numeric(
    quotes_df["quoting_truth"],
    errors="coerce"
).astype("Int64")

quotes_df["quoting_user"] = pd.to_numeric(
    quotes_df["quoting_user"],
    errors="coerce"
).astype("Int64")

# Exclude invalid or unknown quoting users
valid_quotes = quotes_df.loc[
    quotes_df["quoting_truth"].notna()
    & quotes_df["quoting_user"].notna()
    & (quotes_df["quoting_user"] >= 0)
].copy()

# Select events with missing user IDs
unknown_events = events_clean.loc[
    events_clean["user_id"].isna(),
    ["event_id", "event_type", "truth_id"]
].copy()

# Find events whose truth_id appears as quoting_truth
quote_matches = unknown_events.merge(
    valid_quotes[["quoting_truth", "quoting_user"]],
    left_on="truth_id",
    right_on="quoting_truth",
    how="inner"
)

# Report matching events without counting duplicate join rows
matching_event_ids = quote_matches["event_id"].unique()

print("Unknown user events:", len(unknown_events))
print("Events with a matching quoting_truth:", len(matching_event_ids))

print("\nMatching events by event type:")
display(
    unknown_events.loc[
        unknown_events["event_id"].isin(matching_event_ids),
        "event_type"
    ].value_counts()
)

# Check whether each matched event has a unique user candidate
candidate_counts = (
    quote_matches.groupby("event_id")["quoting_user"]
    .nunique()
)

print(
    "\nEvents with exactly one quoting_user candidate:",
    int(candidate_counts.eq(1).sum())
)

print(
    "Events with multiple quoting_user candidates:",
    int(candidate_counts.gt(1).sum())
)


Unknown user events: 106949
Events with a matching quoting_truth: 192

Matching events by event type:


event_type
LI     129
RET     44
REP     19
Name: count, dtype: Int64


Events with exactly one quoting_user candidate: 192
Events with multiple quoting_user candidates: 0


In [18]:

# ============================================================
# STEP 18: VALIDATE UNKNOWN USER HANDLING
# ============================================================

# Identify original sentinel values
original_unknown = (
    pd.to_numeric(
        events_raw["user_id"],
        errors="coerce"
    ) == -1
)

# Verify that sentinel values became missing values
correctly_standardized = (
    events_clean.loc[original_unknown, "user_id"]
    .isna()
    .all()
)

# Verify that known identifiers were preserved
original_known_ids = pd.to_numeric(
    events_raw.loc[~original_unknown, "user_id"],
    errors="coerce"
).astype("Int64")

cleaned_known_ids = events_clean.loc[
    ~original_unknown, "user_id"
].astype("Int64")

known_ids_preserved = original_known_ids.equals(
    cleaned_known_ids
)

# Verify the quality indicator
flags_consistent = (
    events_clean["user_id_unknown"]
    .equals(original_unknown.astype("boolean"))
)

print("Unknown identifiers correctly standardized:",
      correctly_standardized)

print("Known user identifiers preserved:",
      known_ids_preserved)

print("Unknown user flags consistent:",
      flags_consistent)

print(
    "\nRemaining missing user IDs:",
    events_clean["user_id"].isna().sum()
)

assert correctly_standardized
assert known_ids_preserved
assert flags_consistent

print("\nUser ID cleaning successfully validated.")


Unknown identifiers correctly standardized: True
Known user identifiers preserved: True
Unknown user flags consistent: True

Remaining missing user IDs: 106949

User ID cleaning successfully validated.


In [19]:

# ============================================================
# STEP 19: INVESTIGATE EARLY EVENT TIMESTAMPS
# ============================================================

early_events = events_clean.loc[
    before_platform_mask,
    [
        "event_id",
        "event_type",
        "truth_id",
        "user_id",
        "timestamp"
    ]
].copy()

# Summarize the temporal range of anomalous events
print("Early events:", len(early_events))

print(
    "\nEarliest event:",
    early_events["timestamp"].min()
)

print(
    "Latest early event:",
    early_events["timestamp"].max()
)

# Check whether these events reference known posts
early_events["truth_in_main"] = (
    early_events["truth_id"].isin(valid_truth_ids)
)

early_events["truth_in_additional"] = (
    early_events["truth_id"].isin(additional_truth_ids)
)

print("\nReference dataset coverage:")

display(
    early_events[
        ["truth_in_main", "truth_in_additional"]
    ]
    .value_counts()
    .reset_index(name="records")
)

# Show only records requiring further investigation
display(
    early_events.sort_values("timestamp").head(10)
)


Early events: 51

Earliest event: 2022-02-14 00:00:00+00:00
Latest early event: 2022-02-20 12:38:47.653886370+00:00

Reference dataset coverage:


,truth_in_main,truth_in_additional,records
0,True,False,51


,event_id,event_type,truth_id,user_id,timestamp,truth_in_main,truth_in_additional
474706,101746,CR,719187,16675,2022-02-14 00:00:00.000000000+00:00,True,False
553970,7944299,LI,896293,71919,2022-02-14 19:18:26.402903554+00:00,True,False
540173,7934640,LI,896293,3761,2022-02-14 23:10:19.134511123+00:00,True,False
686088,18851822,CR,34406,826,2022-02-14 23:53:00.000000000+00:00,True,False
663079,28738604,RET,325285,16675,2022-02-15 02:22:16.055717891+00:00,True,False
680823,29904889,RET,366261,455,2022-02-15 03:03:34.693122522+00:00,True,False
613341,1777551,LI,754447,12967,2022-02-15 12:40:43.652638992+00:00,True,False
618088,22187,LI,807654,10431,2022-02-15 12:54:19.330762575+00:00,True,False
628738,32996227,LI,510629,12738,2022-02-15 15:52:39.235324348+00:00,True,False
712098,28733473,RET,325285,27,2022-02-15 16:26:42.942436421+00:00,True,False


In [20]:

# ============================================================
# STEP 20: VALIDATE EARLY EVENTS AGAINST POST TIMESTAMPS
# ============================================================

# Load only the fields required for temporal validation
truth_timestamps = pd.read_csv(
    TRUTHS_PATH,
    usecols=["id", "timestamp"],
    dtype={"id": "string", "timestamp": "string"}
)

# Standardize identifiers and timestamps
truth_timestamps["id"] = pd.to_numeric(
    truth_timestamps["id"],
    errors="coerce"
).astype("Int64")

truth_timestamps["timestamp"] = pd.to_datetime(
    truth_timestamps["timestamp"],
    errors="coerce",
    utc=True,
    format="mixed"
)

# Build a reference table with one row per truth_id
# Retain the full timestamp range when duplicate IDs exist
truth_timestamp_reference = (
    truth_timestamps
    .groupby("id", as_index=False)
    .agg(
        earliest_post_timestamp=("timestamp", "min"),
        latest_post_timestamp=("timestamp", "max"),
        distinct_post_timestamps=("timestamp", "nunique")
    )
)

# Join only the 51 early events
early_temporal_check = early_events.merge(
    truth_timestamp_reference,
    left_on="truth_id",
    right_on="id",
    how="left",
    validate="many_to_one"
)

# Classify temporal consistency
early_temporal_check["temporal_status"] = "consistent"

early_temporal_check.loc[
    early_temporal_check["earliest_post_timestamp"].isna(),
    "temporal_status"
] = "missing_post_timestamp"

early_temporal_check.loc[
    early_temporal_check["timestamp"]
    < early_temporal_check["earliest_post_timestamp"],
    "temporal_status"
] = "event_before_post"

early_temporal_check.loc[
    (
        early_temporal_check["timestamp"]
        >= early_temporal_check["earliest_post_timestamp"]
    )
    & (
        early_temporal_check["timestamp"]
        < early_temporal_check["latest_post_timestamp"]
    ),
    "temporal_status"
] = "ambiguous_duplicate_timestamps"

# Report
print("Temporal consistency results:")

display(
    early_temporal_check["temporal_status"]
    .value_counts()
    .rename_axis("status")
    .reset_index(name="records")
)

print("\nEvents requiring investigation:")

display(
    early_temporal_check.loc[
        early_temporal_check["temporal_status"] != "consistent",
        [
            "event_id",
            "event_type",
            "truth_id",
            "timestamp",
            "earliest_post_timestamp",
            "latest_post_timestamp",
            "temporal_status"
        ]
    ].head(20)
)

assert len(early_temporal_check) == len(early_events)


Temporal consistency results:


,status,records
0,consistent,51



Events requiring investigation:


,event_id,event_type,truth_id,timestamp,earliest_post_timestamp,latest_post_timestamp,temporal_status


In [21]:

# ============================================================
# STEP 21: GLOBAL EVENT-POST TEMPORAL VALIDATION
# ============================================================

# Validate all events whose posts exist in truths.csv
main_truth_events = events_clean.loc[
    events_clean["truth_id"].isin(valid_truth_ids),
    ["event_id", "event_type", "truth_id", "timestamp"]
].copy()

# Join post creation timestamps
global_temporal_check = main_truth_events.merge(
    truth_timestamp_reference,
    left_on="truth_id",
    right_on="id",
    how="left",
    validate="many_to_one"
)

# Define temporal consistency conditions
missing_post_timestamp = (
    global_temporal_check["earliest_post_timestamp"].isna()
)

event_before_post = (
    global_temporal_check["timestamp"]
    < global_temporal_check["earliest_post_timestamp"]
)

ambiguous_timestamp = (
    global_temporal_check["timestamp"]
    >= global_temporal_check["earliest_post_timestamp"]
) & (
    global_temporal_check["timestamp"]
    < global_temporal_check["latest_post_timestamp"]
)

# Classify results
global_temporal_check["temporal_status"] = "consistent"

global_temporal_check.loc[
    missing_post_timestamp,
    "temporal_status"
] = "missing_post_timestamp"

global_temporal_check.loc[
    ambiguous_timestamp,
    "temporal_status"
] = "ambiguous_duplicate_timestamps"

global_temporal_check.loc[
    event_before_post,
    "temporal_status"
] = "event_before_post"

# Display summary
print("Global temporal consistency report:")

display(
    global_temporal_check["temporal_status"]
    .value_counts()
    .rename_axis("status")
    .reset_index(name="records")
)

print("\nPotential temporal inconsistencies:")

display(
    global_temporal_check.loc[
        global_temporal_check["temporal_status"] != "consistent",
        [
            "event_id",
            "event_type",
            "truth_id",
            "timestamp",
            "earliest_post_timestamp",
            "latest_post_timestamp",
            "temporal_status"
        ]
    ].head(15)
)

assert len(global_temporal_check) == len(main_truth_events)

print("\nEvents validated:", len(global_temporal_check))
print("Original records preserved:", len(events_clean))


Global temporal consistency report:


,status,records
0,consistent,482218
1,event_before_post,1567



Potential temporal inconsistencies:


,event_id,event_type,truth_id,timestamp,earliest_post_timestamp,latest_post_timestamp,temporal_status
405774,35579087,LI,1269331,2024-07-07 23:59:59.999999+00:00,2024-07-08 17:47:00+00:00,2024-07-08 17:47:00+00:00,event_before_post
405781,37881283,CR,1966248,2024-08-23 23:59:59.999999+00:00,2024-08-24 02:38:00+00:00,2024-08-24 02:38:00+00:00,event_before_post
405782,36676055,LI,1558108,2024-07-26 23:59:59.999999+00:00,2024-07-27 12:22:00+00:00,2024-07-27 12:22:00+00:00,event_before_post
405846,39312572,LI,2339153,2024-09-01 23:59:59.999999+00:00,2024-09-02 00:18:00+00:00,2024-09-02 00:18:00+00:00,event_before_post
405861,35315540,LI,1173719,2024-06-24 23:59:59.999999+00:00,2024-06-25 02:18:00+00:00,2024-06-25 02:18:00+00:00,event_before_post
405910,35951189,RET,1378206,2024-07-13 23:59:59.999999+00:00,2024-07-14 01:47:00+00:00,2024-07-14 01:47:00+00:00,event_before_post
405934,38904691,LI,2248154,2024-08-06 23:59:59.999999+00:00,2024-08-07 00:17:00+00:00,2024-08-07 00:17:00+00:00,event_before_post
406048,35384578,LI,1203017,2024-06-27 23:59:59.999999+00:00,2024-06-28 13:21:00+00:00,2024-06-28 13:21:00+00:00,event_before_post
406059,35930490,LI,1372146,2024-07-13 23:59:59.999999+00:00,2024-07-14 15:10:00+00:00,2024-07-14 15:10:00+00:00,event_before_post
406127,37246282,LI,1749435,2024-08-09 23:59:59.999999+00:00,2024-08-10 19:05:00+00:00,2024-08-10 19:05:00+00:00,event_before_post



Events validated: 483785
Original records preserved: 716643


In [22]:

# ============================================================
# STEP 22: INVESTIGATE SYSTEMATIC TIMESTAMP ANOMALIES
# ============================================================

# Select events occurring before their associated posts
temporal_anomalies = global_temporal_check.loc[
    global_temporal_check["temporal_status"] == "event_before_post"
].copy()

# Calculate the time difference in hours
temporal_anomalies["time_difference_hours"] = (
    temporal_anomalies["earliest_post_timestamp"]
    - temporal_anomalies["timestamp"]
).dt.total_seconds() / 3600

# Extract time components
temporal_anomalies["event_hour"] = (
    temporal_anomalies["timestamp"].dt.hour
)

temporal_anomalies["event_minute"] = (
    temporal_anomalies["timestamp"].dt.minute
)

temporal_anomalies["event_second"] = (
    temporal_anomalies["timestamp"].dt.second
)

# Detect timestamps at the very end of the day
end_of_day_mask = (
    (temporal_anomalies["event_hour"] == 23)
    & (temporal_anomalies["event_minute"] == 59)
    & (temporal_anomalies["event_second"] == 59)
)

print("Total temporal anomalies:", len(temporal_anomalies))

print(
    "Anomalies with end-of-day timestamps:",
    int(end_of_day_mask.sum())
)

print(
    "Other temporal anomalies:",
    int((~end_of_day_mask).sum())
)

print("\nTime difference statistics (hours):")

display(
    temporal_anomalies["time_difference_hours"].describe()
)

print("\nAnomalies by event type:")

display(
    temporal_anomalies["event_type"].value_counts()
)


Total temporal anomalies: 1567
Anomalies with end-of-day timestamps: 1567
Other temporal anomalies: 0

Time difference statistics (hours):


count    1.567000e+03
mean     9.196724e+00
std      6.837952e+00
min      2.777778e-10
25%      2.366667e+00
50%      9.783333e+00
75%      1.516667e+01
max      2.371667e+01
Name: time_difference_hours, dtype: float64


Anomalies by event type:


event_type
LI     839
RET    440
REP    183
CR     105
Name: count, dtype: Int64

In [23]:

# ============================================================
# STEP 23: FLAG SYSTEMATIC TEMPORAL ANOMALIES
# ============================================================

# Initialize temporal anomaly indicator
events_clean["temporal_anomaly"] = False

# Identify events with confirmed event-before-post inconsistencies
anomalous_event_ids = set(
    temporal_anomalies["event_id"]
)

# Flag suspicious records without modifying timestamps
events_clean.loc[
    events_clean["event_id"].isin(anomalous_event_ids),
    "temporal_anomaly"
] = True

# Validate flag assignment
flagged_events = int(events_clean["temporal_anomaly"].sum())

print("Flagged temporal anomalies:", flagged_events)

print(
    "Original timestamps preserved:",
    events_clean["timestamp"].equals(
        pd.to_datetime(
            events_raw["timestamp"],
            errors="coerce",
            utc=True,
            format="mixed"
        )
    )
)

print("Records preserved:", len(events_clean))

assert flagged_events == len(temporal_anomalies)
assert len(events_clean) == len(events_raw)

print("\nTemporal anomalies successfully documented.")


Flagged temporal anomalies: 1567
Original timestamps preserved: True
Records preserved: 716643

Temporal anomalies successfully documented.


In [24]:

# ============================================================
# STEP 24: FINAL DATA QUALITY VALIDATION
# ============================================================

# Preserve only the original dataset columns
final_columns = [
    "index",
    "event_id",
    "event_type",
    "truth_id",
    "user_id",
    "timestamp"
]

events_final = events_clean[final_columns].copy()

# Validate record count and schema
assert len(events_final) == len(events_raw)
assert list(events_final.columns) == final_columns

# Validate event identifiers
assert events_final["event_id"].notna().all()
assert events_final["event_id"].is_unique

# Validate event types
assert events_final["event_type"].isin(
    {"LI", "RET", "REP", "CR"}
).all()

# Validate timestamps
assert events_final["timestamp"].notna().all()

# Validate user identifiers
assert (
    events_final["user_id"].isna().sum()
    == 106949
)

assert (
    events_final["user_id"].dropna() >= 0
).all()

# Validate foreign key correspondence
assert events_final["user_id"].dropna().isin(
    valid_user_ids
).all()

assert events_final["truth_id"].isin(
    valid_truth_ids | additional_truth_ids
).all()

# Generate final summary
final_validation_report = pd.DataFrame({
    "check": [
        "Total records",
        "Unique event IDs",
        "Missing user IDs",
        "Missing timestamps",
        "Invalid event types",
        "Final columns"
    ],
    "result": [
        len(events_final),
        events_final["event_id"].nunique(),
        events_final["user_id"].isna().sum(),
        events_final["timestamp"].isna().sum(),
        (~events_final["event_type"].isin(
            {"LI", "RET", "REP", "CR"}
        )).sum(),
        len(events_final.columns)
    ]
})

display(final_validation_report)

print("\nAll final validation checks passed.")
print("Dataset ready for export.")


,check,result
0,Total records,716643
1,Unique event IDs,716643
2,Missing user IDs,106949
3,Missing timestamps,0
4,Invalid event types,0
5,Final columns,6



All final validation checks passed.
Dataset ready for export.


In [25]:
# ============================================================
# STEP 25: LINK EVENTS TO THE INTEGRATED TRUTHS CORPUS
# ============================================================

# Load the ARFF-to-integrated Truth ID mapping generated
# in the additional Truths integration notebook.

MAPPING_PATH = DATA_DIR / "arff_truth_id_mapping.csv"
TRUTHS_CLEAN_PATH = DATA_DIR / "truths_clean.csv"

for path in [MAPPING_PATH, TRUTHS_CLEAN_PATH]:
    if not path.is_file():
        raise FileNotFoundError(f"File not found: {path}")

# ------------------------------------------------------------
# 1. Load and validate the ID mapping
# ------------------------------------------------------------

arff_mapping = pd.read_csv(
    MAPPING_PATH,
    dtype={
        "arff_truth_id": "string",
        "integrated_truth_id": "Int64",
        "mapping_status": "string"
    }
)

arff_mapping["arff_truth_id"] = (
    arff_mapping["arff_truth_id"].str.strip()
)

assert arff_mapping["arff_truth_id"].is_unique
assert arff_mapping["arff_truth_id"].notna().all()
assert arff_mapping["integrated_truth_id"].notna().all()
assert not arff_mapping["mapping_status"].eq(
    "shared_ambiguous"
).any()

print("ARFF mapping successfully loaded.")
print("Mapping records:", len(arff_mapping))

# ------------------------------------------------------------
# 2. Load the cleaned main corpus IDs
# ------------------------------------------------------------

main_truth_ids = pd.read_csv(
    TRUTHS_CLEAN_PATH,
    usecols=["id"],
    dtype={"id": "Int64"}
)["id"]

assert main_truth_ids.notna().all()
assert main_truth_ids.is_unique

main_id_set = set(main_truth_ids.astype("int64"))

print("Cleaned main corpus IDs:", len(main_id_set))

# ------------------------------------------------------------
# 3. Create an enriched Events DataFrame
# ------------------------------------------------------------

# Preserve the original six columns
events_integrated = events_final.copy()

# Add the integrated foreign key
events_integrated["truth_id_integrated"] = pd.Series(
    pd.NA,
    index=events_integrated.index,
    dtype="Int64"
)

# Track the provenance of every mapping
events_integrated["truth_mapping_status"] = pd.Series(
    "unmatched",
    index=events_integrated.index,
    dtype="string"
)

# Preserve original Truth IDs as exact strings for lookup
original_truth_ids = (
    events_integrated["truth_id"].astype("string")
)

# ------------------------------------------------------------
# 4. Match events against the cleaned main corpus
# ------------------------------------------------------------

main_match_mask = (
    events_integrated["truth_id"].isin(main_id_set)
)

events_integrated.loc[
    main_match_mask, "truth_id_integrated"
] = events_integrated.loc[
    main_match_mask, "truth_id"
]

events_integrated.loc[
    main_match_mask, "truth_mapping_status"
] = "main_clean"

# ------------------------------------------------------------
# 5. Recover remaining events using the ARFF mapping
# ------------------------------------------------------------

id_lookup = arff_mapping.set_index(
    "arff_truth_id"
)["integrated_truth_id"]

status_lookup = arff_mapping.set_index(
    "arff_truth_id"
)["mapping_status"]

arff_match_mask = (
    ~main_match_mask
    & original_truth_ids.isin(id_lookup.index)
)

events_integrated.loc[
    arff_match_mask, "truth_id_integrated"
] = (
    original_truth_ids.loc[arff_match_mask]
    .map(id_lookup)
    .astype("Int64")
)

events_integrated.loc[
    arff_match_mask, "truth_mapping_status"
] = (
    "arff_"
    + original_truth_ids.loc[arff_match_mask]
    .map(status_lookup)
    .astype("string")
)

# ------------------------------------------------------------
# 6. Validate referential integrity
# ------------------------------------------------------------

# Valid IDs consist of cleaned main-corpus IDs and
# integrated IDs recorded by the validated ARFF mapping.
valid_integrated_ids = (
    main_id_set
    | set(arff_mapping["integrated_truth_id"].dropna())
)

assert len(events_integrated) == len(events_final)

assert events_integrated["truth_id"].equals(
    events_final["truth_id"]
)

assert events_integrated["event_id"].is_unique

assert events_integrated[
    "truth_id_integrated"
].dropna().isin(valid_integrated_ids).all()

# ------------------------------------------------------------
# 7. Generate the integration report
# ------------------------------------------------------------

print("\nEVENTS-TO-TRUTHS INTEGRATION REPORT")
print("-" * 55)

display(
    events_integrated["truth_mapping_status"]
    .value_counts(dropna=False)
    .rename_axis("mapping_status")
    .reset_index(name="records")
)

print("\nFINAL VALIDATION")
print("Total events:", len(events_integrated))

print(
    "Matched to cleaned main corpus:",
    int(main_match_mask.sum())
)

print(
    "Matched through ARFF mapping:",
    int(arff_match_mask.sum())
)

print(
    "Unresolved Truth references:",
    int(events_integrated["truth_id_integrated"].isna().sum())
)

print(
    "Original Truth IDs preserved:",
    events_integrated["truth_id"].equals(
        events_final["truth_id"]
    )
)

print("\nSAMPLE RECOVERED EVENTS")

display(
    events_integrated.loc[
        arff_match_mask,
        [
            "event_id",
            "truth_id",
            "truth_id_integrated",
            "truth_mapping_status"
        ]
    ].head(10)
)

print("\nSAMPLE UNRESOLVED EVENTS")

display(
    events_integrated.loc[
        events_integrated["truth_id_integrated"].isna(),
        [
            "event_id",
            "truth_id",
            "truth_mapping_status"
        ]
    ].head(10)
)

print("\nIntegration validation completed.")

ARFF mapping successfully loaded.
Mapping records: 35222
Cleaned main corpus IDs: 1038821

EVENTS-TO-TRUTHS INTEGRATION REPORT
-------------------------------------------------------


,mapping_status,records
0,main_clean,483785
1,arff_new_post,232647
2,arff_shared_unique,204
3,arff_shared_canonical_normalized,7



FINAL VALIDATION
Total events: 716643
Matched to cleaned main corpus: 483785
Matched through ARFF mapping: 232858
Unresolved Truth references: 0
Original Truth IDs preserved: True

SAMPLE RECOVERED EVENTS


,event_id,truth_id,truth_id_integrated,truth_mapping_status
405623,131151848,108211822140637685,2418714,arff_new_post
405624,131151849,108211822140637685,2418714,arff_new_post
405625,131151850,108211822140637685,2418714,arff_new_post
405626,131151851,108211822140637685,2418714,arff_new_post
405627,131151852,108211822140637685,2418714,arff_new_post
405628,131151853,108211822140637685,2418714,arff_new_post
405629,131151854,108211822140637685,2418714,arff_new_post
405630,131151855,108211822140637685,2418714,arff_new_post
405631,131151856,108211822140637685,2418714,arff_new_post
405632,131151857,108211822140637685,2418714,arff_new_post



SAMPLE UNRESOLVED EVENTS


,event_id,truth_id,truth_mapping_status



Integration validation completed.


In [26]:
# ============================================================
# STEP 26: EXPORT THE INTEGRATED EVENTS DATASET
# ============================================================

EVENTS_INTEGRATED_PATH = DATA_DIR / "events_integrated.csv"

# Final checks before export
assert len(events_integrated) == len(events_final)

assert events_integrated["event_id"].is_unique

assert events_integrated["truth_id_integrated"].notna().all()

assert events_integrated["truth_id_integrated"].isin(
    valid_integrated_ids
).all()

# Export the enriched dataset
events_integrated.to_csv(
    EVENTS_INTEGRATED_PATH,
    index=False
)

# Verify the exported file
exported_events = pd.read_csv(
    EVENTS_INTEGRATED_PATH,
    dtype={
        "event_id": "Int64",
        "truth_id": "Int64",
        "truth_id_integrated": "Int64",
        "user_id": "Int64"
    }
)

assert len(exported_events) == len(events_integrated)

assert exported_events["event_id"].is_unique

assert exported_events["truth_id_integrated"].equals(
    events_integrated["truth_id_integrated"].reset_index(drop=True)
)

print("INTEGRATED EVENTS EXPORT")
print("-" * 50)

print("File:", EVENTS_INTEGRATED_PATH)
print("Records:", len(exported_events))
print("Columns:", len(exported_events.columns))

print("\nCOLUMNS")
print(exported_events.columns.tolist())

print("\nFILE SIZE (MB)")
print(round(EVENTS_INTEGRATED_PATH.stat().st_size / 1024**2, 2))

print("\nExport validation passed.")

INTEGRATED EVENTS EXPORT
--------------------------------------------------
File: C:\Users\bucco\Desktop\Laboratory of Data Science\project\code_project\data\Project_Data\events_integrated.csv
Records: 716643
Columns: 8

COLUMNS
['index', 'event_id', 'event_type', 'truth_id', 'user_id', 'timestamp', 'truth_id_integrated', 'truth_mapping_status']

FILE SIZE (MB)
62.22

Export validation passed.
